# GetAround - Analyse des retards et pricing ML

**Contexte** : GetAround est une plateforme de location de voitures entre
particuliers. Un probleme recurrent est le retard au checkout : le conducteur
precedent rend la voiture en retard, ce qui impacte la location suivante.
L'objectif est double :

1. Analyser les retards et recommander un seuil minimum entre deux locations
2. Construire un modele ML de pricing deploye via API

**Sommaire** :

1. Analyse des retards (EDA)
2. Simulation de seuils
3. Recommandation
4. Pricing ML - EDA
5. Pricing ML - Modelisation
6. MLflow Tracking
7. Export du modele

## Imports

In [ ]:
# -- Bibliotheque standard --
import os
import logging
import warnings

# -- Bibliotheques tierces --
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import joblib
import mlflow
import mlflow.sklearn
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)

In [ ]:
# Template Plotly fond noir pour export diaporama
# Accent vert GetAround #00B67A
ACCENT = "#00B67A"
ACCENT2 = "#5CE1E6"

pio.templates["dark_pptx"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor="#000000",
        plot_bgcolor="#000000",
        font=dict(color="#FFFFFF", size=14),
        title=dict(font=dict(color="#FFFFFF", size=18)),
        xaxis=dict(
            gridcolor="#222222",
            linecolor="#444444",
            zerolinecolor="#444444",
            tickfont=dict(color="#CCCCCC"),
            title=dict(font=dict(color="#FFFFFF")),
        ),
        yaxis=dict(
            gridcolor="#222222",
            linecolor="#444444",
            zerolinecolor="#444444",
            tickfont=dict(color="#CCCCCC"),
            title=dict(font=dict(color="#FFFFFF")),
        ),
        legend=dict(
            bgcolor="rgba(0,0,0,0)",
            font=dict(color="#FFFFFF"),
        ),
        colorway=[
            ACCENT, ACCENT2, "#FFFFFF",
            "#888888", "#FF6B6B", "#FFD93D",
        ],
    )
)
pio.templates.default = "dark_pptx"

# Dossier d'export des graphes pour le diaporama
os.makedirs("assets/images", exist_ok=True)
print("Template dark_pptx actif")

---
# 1. Analyse des retards (EDA)

## 1.1 Chargement des donnees de retards

In [ ]:
# Chargement du fichier Excel des retards
df_delay = pd.read_excel(
    'data/get_around_delay_analysis.xlsx'
)
df_delay.shape

In [ ]:
# Apercu des premieres lignes
df_delay.head()

In [ ]:
# Types et valeurs non nulles par colonne
df_delay.info()

In [ ]:
# Statistiques descriptives
df_delay.describe()

## 1.2 Valeurs manquantes

In [ ]:
# Comptage des valeurs manquantes par colonne
df_delay.isnull().sum()

In [ ]:
# Analyse des valeurs manquantes :
# - delay_at_checkout : NaN pour les locations annulees
#   (pas de checkout) MAIS aussi pour ~1 700 locations
#   terminees dont le delay n'a pas ete enregistre
# - previous_ended_rental_id et time_delta : NaN quand
#   pas de location precedente sur la meme voiture (91%)

n_canceled = df_delay[
    df_delay['state'] == 'canceled'
].shape[0]
n_ended_no_delay = df_delay[
    (df_delay['state'] == 'ended')
    & df_delay['delay_at_checkout_in_minutes'].isna()
].shape[0]
n_no_prev = df_delay[
    'previous_ended_rental_id'
].isna().sum()

print(f"Locations annulees (sans checkout): {n_canceled}")
print(f"Terminees sans delay: {n_ended_no_delay}")
print(f"Locations sans precedent: {n_no_prev}")

## 1.3 Vue d'ensemble du jeu de donnees

In [ ]:
# Repartition des locations par statut
total = len(df_delay)
ended = df_delay[df_delay['state'] == 'ended']
canceled = df_delay[df_delay['state'] == 'canceled']

print(f"Total: {total}")
print(
    f"Terminees: {len(ended)}"
    f" ({len(ended) / total * 100:.1f}%)"
)
print(
    f"Annulees: {len(canceled)}"
    f" ({len(canceled) / total * 100:.1f}%)"
)

In [ ]:
# Diagramme : repartition par type de checkin
fig = px.pie(
    df_delay,
    names='checkin_type',
    title='Repartition par type de checkin',
    color='checkin_type',
    color_discrete_map={
        'connect': ACCENT,
        'mobile': ACCENT2,
    },
)
fig.show()
fig.write_image(
    'assets/images/checkin_type_pie.png',
    width=1000, height=600, scale=2,
)

In [ ]:
# Diagramme : repartition par statut
fig = px.pie(
    df_delay,
    names='state',
    title='Repartition par statut',
    color='state',
    color_discrete_map={
        'ended': ACCENT,
        'canceled': '#FF6B6B',
    },
)
fig.show()

## 1.4 Analyse des retards au checkout

In [ ]:
# Statistiques sur les retards (locations terminees)
delays = ended['delay_at_checkout_in_minutes'].dropna()
late = delays[delays > 0]
on_time = delays[delays <= 0]

pct_late = len(late) / len(delays) * 100
pct_on_time = len(on_time) / len(delays) * 100

print(f"Locations avec delay renseigne: {len(delays)}")
print(f"En retard (>0 min): {len(late)} ({pct_late:.1f}%)")
print(
    f"A l'heure ou en avance:"
    f" {len(on_time)} ({pct_on_time:.1f}%)"
)
print(
    f"Retard moyen (quand retard): {late.mean():.0f} min"
    f" ({late.mean() / 60:.1f}h)"
)
print(f"Retard median (quand retard): {late.median():.0f} min")

In [ ]:
# Distribution des retards (clippe pour lisibilite)
fig = px.histogram(
    delays.clip(-200, 500),
    nbins=100,
    title=(
        'Distribution des retards au checkout'
    ),
    labels={
        'value': 'Retard (minutes)',
        'count': 'Nombre',
    },
    color_discrete_sequence=[ACCENT],
)
fig.add_vline(
    x=0,
    line_dash='dash',
    line_color='#FF6B6B',
    annotation_text='Heure prevue',
    annotation_font_color='#FF6B6B',
)
fig.show()
fig.write_image(
    'assets/images/delay_distribution.png',
    width=1200, height=600, scale=2,
)

In [ ]:
# Retards par type de checkin
delay_by_type = ended.groupby('checkin_type')[
    'delay_at_checkout_in_minutes'
].agg(['mean', 'median', 'count'])
delay_by_type.columns = [
    'Retard moyen', 'Retard median', 'Nb locations'
]
delay_by_type

In [ ]:
# Boxplot des retards par type de checkin
fig = px.box(
    ended,
    x='checkin_type',
    y='delay_at_checkout_in_minutes',
    title='Distribution des retards par type de checkin',
    range_y=[-200, 500],
    color='checkin_type',
    color_discrete_map={
        'connect': ACCENT,
        'mobile': ACCENT2,
    },
)
fig.show()
fig.write_image(
    'assets/images/delay_by_checkin.png',
    width=1200, height=600, scale=2,
)

**Interpretation de l'EDA retards** :

- Le dataset contient **21 310 locations** dont 84.7% terminees et 15.3% annulees.
- Parmi les locations terminees avec donnees de checkout, **57.5% sont en retard** avec un retard median de 53 min et un retard moyen de 202 min (3.4h). La moyenne est tiree vers le haut par des retards extremes.
- Les locations **Connect** sont en moyenne rendues **en avance** (median -9 min, moyenne -44 min), tandis que les locations **Mobile** sont en moyenne rendues **en retard** (median +14 min, moyenne +88 min). Cela s'explique par le checkin sans contact des voitures Connect qui impose moins de friction au retour.
- Malgre cela, les voitures Connect restent sensibles aux retards car le conducteur suivant ne peut pas contacter le precedent en cas de probleme.
- 4 964 valeurs manquantes sur `delay_at_checkout` : 3 265 annulations + 1 699 locations terminees sans donnee de retard.

---
# 2. Simulation de seuils

## 2.1 Impact sur la location suivante

In [ ]:
# Locations qui ont une location precedente (conflit potentiel)
with_prev = df_delay[
    df_delay['previous_ended_rental_id'].notna()
].copy()
print(
    f"Locations avec location precedente:"
    f" {len(with_prev)}"
    f" ({len(with_prev) / total * 100:.1f}%)"
)

# Recuperer le retard de la location precedente
prev_delays = df_delay.set_index('rental_id')[
    'delay_at_checkout_in_minutes'
]
with_prev['prev_delay'] = (
    with_prev['previous_ended_rental_id'].map(prev_delays)
)

In [ ]:
# Cas problematiques : retard precedent > buffer disponible
col_delta = 'time_delta_with_previous_rental_in_minutes'
problematic = with_prev[
    with_prev['prev_delay'] > with_prev[col_delta]
]
pct_prob = len(problematic) / len(with_prev) * 100
print(
    f"Cas problematiques (retard depasse le buffer):"
    f" {len(problematic)}"
)
print(
    f"Soit {pct_prob:.1f}% des locations consecutives"
)

# Annulations parmi les cas problematiques
prob_canceled = problematic[
    problematic['state'] == 'canceled'
]
pct_canc = len(prob_canceled) / len(problematic) * 100
print(
    f"Dont annulations: {len(prob_canceled)}"
    f" ({pct_canc:.1f}%)"
)

In [ ]:
# Distribution du delta entre locations consecutives
fig = px.histogram(
    with_prev[col_delta],
    nbins=50,
    title=(
        'Distribution du delta entre'
        ' locations consecutives'
    ),
    labels={
        'value': 'Delta (minutes)',
        'count': 'Nombre',
    },
    color_discrete_sequence=[ACCENT],
)
fig.show()

## 2.2 Simulation de differents seuils (threshold)

In [ ]:
# Calcul du compromis pour chaque seuil et scope
thresholds = [0, 15, 30, 60, 120, 180, 240, 360, 720]
results = []

for thresh in thresholds:
    for scope in ['all', 'connect']:
        if scope == 'all':
            subset = with_prev
            prob_sub = problematic
        else:
            subset = with_prev[
                with_prev['checkin_type'] == 'connect'
            ]
            prob_sub = problematic[
                problematic['checkin_type'] == 'connect'
            ]

        if thresh == 0:
            affected = 0
            solved = 0
        else:
            affected = (
                subset[col_delta] < thresh
            ).sum()
            solved = (
                prob_sub[col_delta] + thresh
                >= prob_sub['prev_delay']
            ).sum()

        n_sub = len(subset)
        n_prob = len(prob_sub)
        results.append({
            'threshold_min': thresh,
            'scope': scope,
            'locations_affectees': affected,
            'pct_affectees': (
                round(affected / n_sub * 100, 1)
                if n_sub > 0 else 0
            ),
            'cas_resolus': solved,
            'pct_resolus': (
                round(solved / n_prob * 100, 1)
                if n_prob > 0 else 0
            ),
        })

df_sim = pd.DataFrame(results)
df_sim

In [ ]:
# Graphique de compromis : cas resolus vs locations bloquees
fig = make_subplots(specs=[[{'secondary_y': True}]])

colors = {'all': '#FFFFFF', 'connect': ACCENT}
for scope in ['all', 'connect']:
    sub = df_sim[df_sim['scope'] == scope]
    fig.add_trace(
        go.Scatter(
            x=sub['threshold_min'],
            y=sub['pct_resolus'],
            name=f'Cas resolus ({scope})',
            mode='lines+markers',
            line=dict(color=colors[scope], width=2),
            marker=dict(size=7),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=sub['threshold_min'],
            y=sub['pct_affectees'],
            name=f'Locations bloquees ({scope})',
            mode='lines+markers',
            line=dict(
                color=colors[scope], width=2, dash='dash',
            ),
            marker=dict(size=7),
        ),
        secondary_y=True,
    )

# Marqueur du seuil recommande
fig.add_vline(
    x=120,
    line_dash='dot',
    line_color='#FF6B6B',
    annotation_text='Seuil : 120 min',
    annotation_font_color='#FF6B6B',
)

fig.update_layout(
    title='Compromis : cas resolus vs locations bloquees',
)
fig.update_xaxes(title_text='Seuil (minutes)')
fig.update_yaxes(
    title_text='% cas resolus',
    secondary_y=False,
)
fig.update_yaxes(
    title_text='% locations bloquees',
    secondary_y=True,
)
fig.show()
fig.write_image(
    'assets/images/threshold_tradeoff.png',
    width=1200, height=600, scale=2,
)

---
# 3. Recommandation

**Justification du seuil de simulation** :

Le graphique ci-dessus montre le compromis entre cas resolus et locations bloquees. Le rendement marginal diminue fortement apres 120 min (le gain passe de +12 points entre 60 et 120 min a seulement +8 points entre 120 et 180 min, pour un cout de blocage croissant).

**Seuil recommande : 120 minutes** (scope : Connect uniquement)

- Resout ~84% des cas problematiques pour les voitures Connect
- Ne bloque que ~36% des locations consecutives Connect
- Les voitures Connect representent 20% du parc mais ont un checkin sans contact, donc plus sensibles aux retards
- Un seuil de 120 min est un bon compromis entre protection des utilisateurs et perte de revenus
- Au-dela de 120 min, le gain en cas resolus est marginal alors que le pourcentage de locations bloquees augmente rapidement

---
# 4. Pricing ML - EDA

## 4.1 Chargement et exploration du dataset pricing

In [ ]:
# Chargement du dataset pricing
df_price = pd.read_csv(
    'data/get_around_pricing_project.csv',
    index_col=0,
)
df_price.shape

In [ ]:
# Apercu des premieres lignes
df_price.head()

In [ ]:
# Types et valeurs non nulles
df_price.info()

In [ ]:
# Statistiques descriptives
df_price.describe()

In [ ]:
# Verification des valeurs manquantes
df_price.isnull().sum()

## 4.2 Analyse exploratoire des features

In [ ]:
# Distribution du prix journalier
fig = px.histogram(
    df_price,
    x='rental_price_per_day',
    nbins=50,
    title='Distribution du prix journalier',
    color_discrete_sequence=[ACCENT],
)
fig.show()
fig.write_image(
    'assets/images/price_distribution.png',
    width=1200, height=600, scale=2,
)

In [ ]:
# Prix par type de vehicule
fig = px.box(
    df_price,
    x='car_type',
    y='rental_price_per_day',
    title='Prix par type de vehicule',
    color_discrete_sequence=[ACCENT],
)
fig.show()

In [ ]:
# Prix vs puissance moteur, colore par type de carburant
fig = px.scatter(
    df_price,
    x='engine_power',
    y='rental_price_per_day',
    color='fuel',
    title='Prix vs puissance moteur',
    color_discrete_sequence=[
        ACCENT, ACCENT2, '#FF6B6B', '#FFD93D',
    ],
    opacity=0.6,
)
fig.show()
fig.write_image(
    'assets/images/price_vs_power.png',
    width=1200, height=600, scale=2,
)

In [ ]:
# Prix vs kilometrage, colore par type de vehicule
fig = px.scatter(
    df_price,
    x='mileage',
    y='rental_price_per_day',
    color='car_type',
    title='Prix vs kilometrage',
    color_discrete_sequence=[
        ACCENT, ACCENT2, '#FF6B6B', '#FFD93D',
        '#FFFFFF', '#888888',
    ],
    opacity=0.6,
)
fig.show()

In [ ]:
# Correlation des features numeriques et booleennes
# avec le prix journalier
bool_cols = df_price.select_dtypes(include='bool').columns
df_corr = df_price.copy()
for col in bool_cols:
    df_corr[col] = df_corr[col].astype(int)

corr = (
    df_corr.select_dtypes(include=[np.number])
    .corr()['rental_price_per_day']
    .drop('rental_price_per_day')
    .sort_values(ascending=False)
)
print('Correlations avec le prix :')
print(corr)

In [ ]:
# Diagramme en barres des correlations
fig = px.bar(
    x=corr.values,
    y=corr.index,
    orientation='h',
    title='Correlation des features avec le prix',
    labels={'x': 'Correlation', 'y': 'Feature'},
    color_discrete_sequence=[ACCENT],
)
fig.show()

**Observations correlations** :
- **engine_power** (+0.63) : la puissance moteur est le meilleur predicteur du prix, ce qui est logique (voitures puissantes = gamme superieure)
- **mileage** (-0.45) : le kilometrage fait baisser le prix (usure, decote)
- **automatic_car** (+0.42) : les automatiques sont en general plus cheres
- **has_getaround_connect** (+0.32), **has_gps** (+0.31) : les equipements technologiques augmentent le prix
- **winter_tires** (+0.02) : correlation quasi nulle, les pneus hiver n'influencent pas le prix

---
# 5. Pricing ML - Modelisation

## 5.1 Preprocessing

In [ ]:
# Separation features / target et identification des types
target = 'rental_price_per_day'
X = df_price.drop(columns=[target])
y = df_price[target]

cat_cols = [
    c for c in X.columns
    if X[c].dtype.name in (
        'object', 'str', 'string', 'string[python]'
    )
]
bool_cols = [
    c for c in X.columns
    if X[c].dtype.name == 'bool'
]
num_cols = [
    c for c in X.columns
    if c not in cat_cols and c not in bool_cols
]

# Conversion des booleens en entiers
for col in bool_cols:
    X[col] = X[col].astype(int)

print(f'Categoriques: {cat_cols}')
print(f'Booleens: {bool_cols}')
print(f'Numeriques: {num_cols}')

In [ ]:
# Split train / test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Pipeline de preprocessing :
# - StandardScaler sur les numeriques
# - OneHotEncoder sur les categoriques
# - Passthrough pour les booleens (deja en int)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        (
            'cat',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore',
                sparse_output=False,
            ),
            cat_cols,
        ),
        ('bool', 'passthrough', bool_cols),
    ]
)

## 5.2 Regression lineaire (baseline)

In [ ]:
# Regression lineaire (baseline)
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression()),
])
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)

lr_r2 = r2_score(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

print("=== Regression lineaire ===")
print(f"R2:   {lr_r2:.4f}")
print(f"MAE:  {lr_mae:.2f} EUR")
print(f"RMSE: {lr_rmse:.2f} EUR")

**Interpretation Regression lineaire** : le R2 d'environ 0.69 indique que le modele lineaire explique 69% de la variance du prix. La MAE (~12 EUR) et la RMSE (~18 EUR) montrent que les predictions sont en moyenne a 12 EUR du prix reel. C'est un baseline correct mais ameliorable : la relation entre les features et le prix n'est probablement pas strictement lineaire.

## 5.3 Gradient Boosting (modele retenu)

In [ ]:
# Gradient Boosting (modele retenu)
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
    )),
])
gb_pipe.fit(X_train, y_train)
gb_pred = gb_pipe.predict(X_test)

gb_r2 = r2_score(y_test, gb_pred)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))

print("=== Gradient Boosting ===")
print(f"R2:   {gb_r2:.4f}")
print(f"MAE:  {gb_mae:.2f} EUR")
print(f"RMSE: {gb_rmse:.2f} EUR")

**Interpretation Gradient Boosting** : le R2 de ~0.75 est nettement superieur au baseline lineaire (+9 points). La MAE de ~10 EUR signifie que le modele se trompe en moyenne de 10 EUR sur le prix journalier (pour un prix moyen de 121 EUR, soit ~8% d'erreur). La RMSE de ~16 EUR est plus elevee que la MAE car elle penalise davantage les grosses erreurs. Ce modele est retenu pour le deploiement.

## 5.4 Validation croisee

In [ ]:
# Validation croisee (5 folds)
# Note : les warnings "unknown categories" sont attendus
# car certaines categories rares (model_key, car_type)
# n'apparaissent pas dans tous les folds.
# handle_unknown='ignore' les gere.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=UserWarning,
        module="sklearn",
    )
    cv_scores = cross_val_score(
        gb_pipe, X, y, cv=5, scoring='r2'
    )

print(
    f'CV R2: {cv_scores.mean():.4f}'
    f' (+/- {cv_scores.std():.4f})'
)
print(
    f'Scores par fold:'
    f' {[round(s, 4) for s in cv_scores]}'
)

**Analyse CV vs Test** : le R2 en cross-validation (0.693) est inferieur au R2 sur le test set (0.750), ce qui indique un **leger overfitting** du Gradient Boosting. L'ecart reste modere (~0.06 points) et la variance entre folds (0.58 a 0.77) reflete l'heterogeneite des donnees. Le modele reste fiable pour une utilisation en production avec une erreur moyenne de ~10 EUR.

---
# 6. MLflow Tracking

In [ ]:
# Suppression des warnings informatifs MLflow
warnings.filterwarnings(
    "ignore", category=FutureWarning, module="mlflow"
)
logging.getLogger("mlflow.sklearn").setLevel(
    logging.ERROR
)

# Configuration du tracking local
mlflow.set_tracking_uri(
    "file:///"
    + os.path.abspath("mlruns").replace("\\", "/")
)
mlflow.set_experiment("getaround_pricing")

# --- Run 1 : Regression lineaire ---
with mlflow.start_run(run_name="linear_regression"):
    lr_pipe_mlf = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ])
    lr_pipe_mlf.fit(X_train, y_train)
    lr_pred_mlf = lr_pipe_mlf.predict(X_test)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_metric(
        "r2", r2_score(y_test, lr_pred_mlf)
    )
    mlflow.log_metric(
        "mae", mean_absolute_error(y_test, lr_pred_mlf)
    )
    mlflow.log_metric(
        "rmse",
        np.sqrt(mean_squared_error(y_test, lr_pred_mlf)),
    )
    mlflow.sklearn.log_model(lr_pipe_mlf, name="model")

# --- Run 2 : Gradient Boosting ---
with mlflow.start_run(run_name="gradient_boosting"):
    gb_pipe_mlf = Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            random_state=42,
        )),
    ])
    gb_pipe_mlf.fit(X_train, y_train)
    gb_pred_mlf = gb_pipe_mlf.predict(X_test)

    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric(
        "r2", r2_score(y_test, gb_pred_mlf)
    )
    mlflow.log_metric(
        "mae", mean_absolute_error(y_test, gb_pred_mlf)
    )
    mlflow.log_metric(
        "rmse",
        np.sqrt(mean_squared_error(y_test, gb_pred_mlf)),
    )
    mlflow.sklearn.log_model(gb_pipe_mlf, name="model")

print("Runs MLflow enregistres avec succes")

**MLflow** : deux runs enregistres dans l'experience `getaround_pricing`. Le premier (linear_regression) logue les parametres, les metriques (R2, MAE, RMSE) et le pipeline sklearn complet. Le second (gradient_boosting) ajoute les hyperparametres specifiques (n_estimators=200, max_depth=5, learning_rate=0.1). Le tracking local est dans `mlruns/`, consultable via `mlflow ui`.

---
# 7. Export du modele

## 7.1 Feature importance

In [ ]:
# Importance des features dans le Gradient Boosting
ohe_features = list(
    gb_pipe.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(cat_cols)
)
feature_names = num_cols + ohe_features + bool_cols

importances = (
    gb_pipe.named_steps['model'].feature_importances_
)
fi = pd.Series(
    importances, index=feature_names
).sort_values(ascending=True)

fig = px.bar(
    x=fi.values,
    y=fi.index,
    orientation='h',
    title='Feature importance (Gradient Boosting)',
    labels={'x': 'Importance', 'y': 'Feature'},
    color_discrete_sequence=[ACCENT],
)
fig.update_layout(height=600)
fig.show()
fig.write_image(
    'assets/images/feature_importance.png',
    width=1200, height=700, scale=2,
)

In [ ]:
# Graphique predictions vs valeurs reelles
fig = px.scatter(
    x=y_test,
    y=gb_pred,
    labels={
        'x': 'Prix reel (EUR)',
        'y': 'Prix predit (EUR)',
    },
    title='Predictions vs valeurs reelles',
    color_discrete_sequence=[ACCENT],
    opacity=0.6,
)
fig.add_trace(go.Scatter(
    x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()],
    mode='lines',
    line=dict(color='#FF6B6B', dash='dash', width=2),
    name='y = x',
))
fig.show()
fig.write_image(
    'assets/images/predictions_vs_actual.png',
    width=1200, height=600, scale=2,
)

## 7.2 Sauvegarde du pipeline

In [ ]:
# Export du modele pour l'API
joblib.dump(gb_pipe, 'api/model.joblib')
print('Modele exporte dans api/model.joblib')

In [ ]:
# Verification du modele exporte
loaded_model = joblib.load('api/model.joblib')
sample = X_test.iloc[:3]
preds = loaded_model.predict(sample)
print("Test de prediction :")
for i in range(3):
    print(
        f"  Predit: {preds[i]:.0f} EUR"
        f" | Reel: {y_test.iloc[i]} EUR"
    )

---
## Synthese

**Analyse des retards :**
- 57.5% des locations (avec donnees de checkout) se terminent en retard, avec un retard median de 53 min
- Les voitures Connect sont en moyenne rendues en avance (-44 min) vs Mobile en retard (+88 min)
- 218 cas problematiques identifies sur 1 841 locations consecutives (11.8%), dont 37 annulations (17%)
- **Recommandation** : seuil de 120 min, scope Connect --> resout 84% des conflits, bloque 36% des locations consecutives Connect

**Pricing ML :**
- Gradient Boosting retenu : R2=0.75 (test), MAE=10.29 EUR, RMSE=16.22 EUR
- CV R2=0.69 (+/- 0.07) --> leger overfitting mais performances stables
- Features les plus importantes : puissance moteur (46%), kilometrage (27%), GPS (5%)
- Modele exporte en pipeline sklearn et deploye via API FastAPI sur HuggingFace Spaces